In [11]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import torch
import numpy as np
import os
import sys
sys.path.append(os.path.abspath('../..'))
from utils.ps_scan_motif_finder import run_ps_scan_motif_finder_dask
import dotenv


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
#GET NECESSARY .ENV VARIABLES
dotenv.load_dotenv('../../.env')
SWEEP_EXP_DIR=os.getenv("SWEEP_EXP_DIR")
SWEEP_ANALYSIS_DIR=os.getenv("SWEEP_ANALYSIS_DIR")

In [13]:
def get_best_model_path(train_set="hpa_uniprot_combined_trainset", level=1):
    avg_metrics = pd.read_csv(f"{SWEEP_ANALYSIS_DIR}/overall_metrics.csv")

    #DROP DUPLICATE RUNS
    idx = avg_metrics.groupby(
        ["exp_name",
        "category_level",
        "metadata_file",
        "clip_len",
        "agg_method",
        "mlp_dropout",
        "loss"
        ])["macro_ap"].idxmax()
    avg_metrics = avg_metrics.loc[idx].reset_index(drop=True)


    #GET BEST MODEL
    best_model = avg_metrics.iloc[avg_metrics[
        (avg_metrics.metadata_file == train_set)  &
        (avg_metrics.category_level == f"level{level}")
    ].macro_ap.idxmax()]

    run_id = best_model.run_id
    plm = best_model.exp_name

    #PATH TO RUN WITH BEST MODEL
    path_to_best_model = f"{SWEEP_EXP_DIR}/{plm}_hpa_uniprot_combined_trainset/{run_id}"

    return path_to_best_model, best_model

PATH_TO_BEST_MODEL, best_model = get_best_model_path()

In [17]:
! $PATH_TO_BEST_MODEL

/bin/bash: /scratch/groups/emmalu/seq2loc/sweep_experiments/ProtT5_hpa_uniprot_combined_trainset/ze5s13k3: Is a directory


## Find attention peaks in all HOU proteins

In [20]:
# run script
if not os.path.exists("../../datasets/intermediate/motif/hou_with_peak_positions.csv"):
    ! python ../../utils/attention_peak_finding.py \
        --input_file ../../datasets/final/hou_testset.csv \
        --best_model $PATH_TO_BEST_MODEL \
        --output_file ../../datasets/intermediate/motif/hou_with_peak_positions.csv

/home/groups/emmalu/zwefers/seq2loc/notebooks/analysis/../../utils/attention_peak_finding.py:201: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  att = torch.load(f'{folder}fo

## Scrape Prosite motifs in HOU proteins

In [21]:
hou = pd.read_csv('../../datasets/intermediate/motif/hou_with_peak_positions.csv')
hou.head()

,uniprot_id,ensembl_id,level1,level2,level3,sequence,From,Entry,Reviewed,Entry Name,...,Developmental stage,Induction,Tissue specificity,Transit peptide,Signal peptide,3D,Beta strand,Helix,Turn,peak_positions
0,A0A087X0K9,ENSG00000104067,plasma-membrane,plasma-membrane,plasma-membrane,MSARAAAAKSTAMEETAIWEQHTVTLHRAPGFGFGIAISGGRDNPH...,A0A087X0K9,A0A087X0K9,unreviewed,A0A087X0K9_HUMAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0, 50], [0, 50], [147, 197], [162, 212], [39..."
1,A0A0C4DGS1,ENSG00000244038,endoplasmic-reticulum,endoplasmic-reticulum,endomembrane-system,MEPSTAARAWALFWLLLPLLGAVCASGPRTLVLLDNLNVRETHSLF...,A0A0C4DGS1,A0A0C4DGS1,unreviewed,A0A0C4DGS1_HUMAN,...,NaN,NaN,NaN,NaN,"SIGNAL 1..25; /evidence=""ECO:0000256|RuleBase:...",NaN,NaN,NaN,NaN,"[[0, 50], [389, 439]]"
2,A0A1B0GTU4,ENSG00000089159,plasma-membrane,plasma-membrane,plasma-membrane,MDDLDALLADLESTTSHISKRPVFLSEETPYSYPTGNHTYQEIAVP...,A0A1B0GTU4,A0A1B0GTU4,unreviewed,A0A1B0GTU4_HUMAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0, 50], [95, 145], [110, 160], [121, 171], [..."
3,A0A2R8YG42,ENSG00000102606,plasma-membrane,plasma-membrane,plasma-membrane,MNSAEQTVTWLITLGVLESPKKTISDPEGFLQASLKDGVVLCRLLE...,A0A2R8YG42,A0A2R8YG42,unreviewed,A0A2R8YG42_HUMAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0, 50], [29, 79], [120, 170], [128, 178], [1..."
4,A0A6Q8PGB0,ENSG00000181222,nucleoplasm,nucleus,nucleus,MHGGGPPSGDSACPLRTIKRVQFGVLSPDELKRMSVTEGGIKYPET...,A0A6Q8PGB0,A0A6Q8PGB0,unreviewed,A0A6Q8PGB0_HUMAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0, 50], [151, 201], [1485, 1535], [1507, 155..."


In [22]:
motifs_df = run_ps_scan_motif_finder_dask(hou)
motifs_df.head()
motifs_df.to_csv('../../datasets/intermediate/motif/hou_prosite_motifs.csv', index=False)

Starting ScanProsite (Dask) for 3814 proteins with up to 3 concurrent workers...
[                                        ] | 0% Completed | 27.92 sms


KeyboardInterrupt: 